# Prefix programs: Colab runner

Runs the seminar grid on a Colab GPU. Every script is resumable: rerun a cell after a disconnect and it continues from `results/*.csv`.

**Before running:** set `GITHUB_TOKEN` (fine-grained PAT with read access to the repo) and `HF_TOKEN` (read) in Colab's Secrets panel, and accept the FLORES+ terms on the Hub once.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/prefix-programs'
GH = userdata.get('GITHUB_TOKEN')
!git clone -q https://{GH}@github.com/sirotenkogeorgii/prefix-programs.git /content/prefix-programs 2>/dev/null || (cd /content/prefix-programs && git pull -q)
%cd /content/prefix-programs
!bash scripts/colab_bootstrap.sh

## 1. Pilot on `dev` (fixes operating points and the k grid before any devtest number)

Short runs on one cell per task to check that bits fall with k, pick the prompt-tuning learning rate, and time one run.

In [ ]:
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru \
    --ks 1,4,16 --seeds 0 --steps 300 --eval-every 50 --baselines --max-test 400 \
    --results results/pilot.csv --ckpt-dir checkpoints/pilot

In [ ]:
import pandas as pd
df = pd.read_csv('results/pilot.csv')
df[['cell','design','k','dev_bits_per_token','test_bits_per_token','test_bits_per_token_noncopy','best_step']]

## 2. Design A sweep, baselines and LoRA floors (WP3)

8 cells × k ∈ {1,2,4,8,16,32} × 2 seeds, plus `lang` and `task` conditioning at one seed.

In [ ]:
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 1,2,4,8,16,32 --seeds 0,1 --conds none --baselines --floors --results results/sweep.csv

In [ ]:
# Conditioning: prefixes on top of the language label (lang) and the task instruction (task), one seed.
# 'both' is the natural-language baseline only (no prefix), as preregistered.
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 1,2,4,8,16,32 --seeds 0 --conds lang,task --baselines --floors --results results/sweep.csv
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks "" --seeds 0 --conds both --baselines --results results/sweep.csv


## 3. Variational rates (WP4a): flat at k* ± neighbours, then hierarchical

Fill `--ks` from the k* table of step 2 (see `prefixlab.analysis.curves.k_star`).

In [ ]:
!python scripts/run_sweep.py --design flat --betas 0.3,1,3 --model Qwen/Qwen2.5-1.5B \
    --tasks translation,recall,continuation --langs ru,de,en --ks 2,4,8 --seeds 0 --results results/flat.csv

In [ ]:
!python scripts/run_hier.py --group task --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 2,4,8 --betas 0.3,1,3 --taus 1 --results results/hier_task.csv
!python scripts/run_hier.py --group lang --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 2,4,8 --betas 0.3,1,3 --taus 1 --results results/hier_lang.csv

## 4. Composition, distances, regime 2, mechanism (WP4b/WP5)

Scripts for these land in `scripts/` as they are written; each follows the same resumable pattern.